In [19]:
# Moteur de recherche en HTML pur dans Jupyter
from IPython.display import HTML

html = """
<div style="font-family: Arial; padding: 20px; background: #1a1a2e; color: white; border-radius: 10px;">
    <h2>🎬 Moteur de recherche de films</h2>
    <input id="search" type="text" placeholder="Rechercher un film..." 
           style="width:300px; padding:8px; border-radius:5px; border:none; margin-right:10px">
    <select id="langue" style="padding:8px; border-radius:5px; border:none; margin-right:10px">
        <option value="">Toutes les langues</option>
        <option value="en">Anglais</option>
        <option value="fr">Français</option>
        <option value="es">Espagnol</option>
        <option value="ja">Japonais</option>
        <option value="ko">Coréen</option>
    </select>
    <button onclick="rechercher()" 
            style="padding:8px 15px; background:#e94560; border:none; color:white; border-radius:5px; cursor:pointer">
        Rechercher
    </button>
    <div id="results" style="margin-top:20px"></div>
</div>

<script>
async function rechercher() {
    const texte = document.getElementById('search').value;
    const langue = document.getElementById('langue').value;
    
    let query = {
        query: {
            bool: {
                must: texte ? [{bool: {should: [
                    {match: {title: texte}},
                    {match: {overview: texte}}
                ], minimum_should_match: 1}}] : [{match_all: {}}],
                filter: langue ? [{term: {original_language: langue}}] : []
            }
        },
        sort: [{popularity: "desc"}],
        size: 10
    };
    
    const response = await fetch('http://localhost:9200/movies_clean/_search', {
        method: 'POST',
        headers: {'Content-Type': 'application/json'},
        body: JSON.stringify(query)
    });
    
    const data = await response.json();
    const hits = data.hits.hits;
    const total = data.hits.total.value;
    
    let html = `<p>🎬 <strong>${total} films trouvés</strong></p>`;
    
    hits.forEach(hit => {
        const src = hit._source;
        html += `
            <div style="background:#16213e; padding:15px; margin:10px 0; border-radius:8px; border-left:4px solid #e94560">
                <h3 style="margin:0; color:#e94560">${src.title || 'Sans titre'}</h3>
                <p style="margin:5px 0">⭐ ${src.vote_average}/10 | 🌍 ${src.original_language} | 📅 ${src.release_date ? src.release_date.substring(0,10) : 'N/A'}</p>
                <p style="margin:0; color:#aaa; font-size:0.9em">${src.overview ? src.overview.substring(0,200) + '...' : ''}</p>
            </div>
        `;
    });
    
    document.getElementById('results').innerHTML = html;
}
</script>
"""

display(HTML(html))